# HealthDemand
## Análisis de Demanda Médica — Diagnóstico histórico de turnos
### Período analizado: sep 2025 – ago 2026  ·  50,000 turnos  ·  6 especialidades  ·  3 sedes


![](https://drive.google.com/uc?id=1rGm7a1nfISfIBnseC5_9D4OvC1bbUitF)

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go

df = pd.read_csv('healthdemand.csv')

# 1. Convertir a datetime (las que no se puedan parsear quedan como NaT)
df['fecha_hora_turno'] = pd.to_datetime(df['fecha_hora_turno'], errors='coerce')
df['fecha_solicitud']  = pd.to_datetime(df['fecha_solicitud'],  errors='coerce')

# 2. Eliminar filas donde alguna de esas dos columnas sea NaT
df = df.dropna(subset=['fecha_hora_turno', 'fecha_solicitud'])

# 3. (Opcional) Resetear el índice
df = df.reset_index(drop=True)
df = df.dropna()

df1 = df.copy()

df1['fecha_hora_turno'] = pd.to_datetime(df1['fecha_hora_turno'], errors='coerce')
df1['fecha_solicitud']  = pd.to_datetime(df1['fecha_solicitud'],  errors='coerce')

df1['fecha_turno']         = df1['fecha_hora_turno'].dt.date
df1['hora_turno']          = df1['fecha_hora_turno'].dt.time
df1['fecha_solicitud_dia'] = df1['fecha_solicitud'].dt.date
df1['hora_solicitud']      = df1['fecha_solicitud'].dt.time

df1['dias_espera'] = (df1['fecha_hora_turno'] - df1['fecha_solicitud']).dt.total_seconds() / 86400
df1 = df1[df1['dias_espera'] >= 0]

# Conteo de turnos por estado
top_estado = (df['estado_turno'].value_counts().head(5).reset_index())
top_estado.columns = ['estado_turno', 'cantidad']

top_estado = (df['id_profesional'].value_counts().head(5).reset_index())
top_pares = (df[['id_profesional', 'estado_turno']].value_counts().tail(5).reset_index(name='cantidad'))

# Filtrar solo turnos con estado relevante (excluir Cancelado)
df_relevante = df[df1['estado_turno'].isin(['Atendido', 'Ausente'])].copy()

# Agrupar por profesional y calcular métricas
resumen = df_relevante.groupby(['id_profesional', 'especialidad']).agg(total_turnos=('id_turno', 'count'),atendidos=('estado_turno', lambda x: (x == 'Atendido').sum()), ausentes=('estado_turno', lambda x: (x == 'Ausente').sum())).reset_index()
resumen['tasa_ausentismo'] = (resumen['ausentes'] / resumen['total_turnos'] * 100).round(2)
min_turnos = 20
resumen_filtrado = resumen[resumen['total_turnos'] >= min_turnos].copy()
top_ausentismo = resumen_filtrado.sort_values('tasa_ausentismo', ascending=False)

top5 = top_ausentismo.head(5).copy()
top5['etiqueta'] = top5['id_profesional'].astype(str) + ' — ' + top5['especialidad']

In [ ]:
# Eliminar filas sin fecha de solicitud válida
df = df.dropna(subset=['fecha_solicitud'])

# AGREGACIÓN MENSUAL
df['anio_mes'] = df['fecha_solicitud'].dt.to_period('M').astype(str)

demanda_mensual = (df.groupby('anio_mes').size().reset_index(name='volumen_demanda').sort_values('anio_mes'))

# TENDENCIA (regresión lineal sobre el índice temporal)

x = np.arange(len(demanda_mensual))
y = demanda_mensual['volumen_demanda'].values

coef = np.polyfit(x, y, 1)
tendencia = np.polyval(coef, x)

demanda_mensual['tendencia'] = tendencia
demanda_mensual['variacion_pct'] = (demanda_mensual['volumen_demanda'].pct_change() * 100).round(2)

In [ ]:
for col in df.columns:
  print(f"valores únicos en la columna'{col}': {df[col].nunique()}")
  if df[col].nunique() < 50:
    print(df[col].unique())
    print('-' * 50)

print(df.info())
print(df1[['id_turno','fecha_turno','hora_turno',
           'fecha_solicitud_dia','hora_solicitud']].head())
print(df1.info())

print(f"Total de registros válidos: {len(df1)}")
print(f"Días de espera promedio global: {df1['dias_espera'].mean():.2f} días")
print(f"Mediana: {df1['dias_espera'].median():.2f} días")
print(f"Desviación estándar: {df1['dias_espera'].std():.2f} días")
print(top_estado)
print(top_pares.tail(5))
print(f"Total de turnos en el dataset: {len(df):,}")
print(f"Turnos relevantes (Atendido/Ausente): {len(df_relevante):,}")
print(f"Turnos cancelados (excluidos): {len(df1[df['estado_turno'] == 'Cancelado']):,}")
print("=" * 100)
print(f"TOP 5 PROFESIONALES CON MAYOR AUSENTISMO (mínimo {min_turnos} turnos)")
print("=" * 100)
print(top_ausentismo.head(5).to_string(index=False))
print(top_ausentismo.info())
print(demanda_mensual)

valores únicos en la columna'id_turno': 50000
valores únicos en la columna'fecha_hora_turno': 16512
valores únicos en la columna'fecha_solicitud': 17400
valores únicos en la columna'especialidad': 6
['Oncología' 'Traumatología' 'Pediatría' 'Clínica Médica' 'Psiquiatría'
 'Cardiología']
--------------------------------------------------
valores únicos en la columna'sede': 3
['Sede Norte' 'Sede Centro' 'Sede Sur']
--------------------------------------------------
valores únicos en la columna'id_profesional': 49
['MED-6' 'MED-47' 'MED-38' 'MED-17' 'MED-7' 'MED-26' 'MED-5' 'MED-3'
 'MED-43' 'MED-20' 'MED-15' 'MED-12' 'MED-39' 'MED-4' 'MED-49' 'MED-46'
 'MED-18' 'MED-34' 'MED-8' 'MED-16' 'MED-27' 'MED-42' 'MED-37' 'MED-2'
 'MED-23' 'MED-25' 'MED-45' 'MED-19' 'MED-21' 'MED-10' 'MED-32' 'MED-9'
 'MED-35' 'MED-44' 'MED-22' 'MED-36' 'MED-41' 'MED-28' 'MED-14' 'MED-48'
 'MED-30' 'MED-11' 'MED-13' 'MED-33' 'MED-24' 'MED-29' 'MED-31' 'MED-40'
 'MED-1']
--------------------------------------------

In [ ]:
print(df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 9 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   id_turno          50000 non-null  int64         
 1   fecha_hora_turno  50000 non-null  datetime64[ns]
 2   fecha_solicitud   50000 non-null  datetime64[ns]
 3   especialidad      50000 non-null  object        
 4   sede              50000 non-null  object        
 5   id_profesional    50000 non-null  object        
 6   tipo_atencion     50000 non-null  object        
 7   estado_turno      50000 non-null  object        
 8   anio_mes          50000 non-null  object        
dtypes: datetime64[ns](2), int64(1), object(6)
memory usage: 3.4+ MB
None


### Resumen ejecutivo

**Indicadores clave del período sep 2025 – ago 2026**

In [ ]:
print(f"Total de registros válidos: {len(df1)}")
print(f"Días de espera promedio global: {df1['dias_espera'].mean():.2f} días")

Total de registros válidos: 50000
Días de espera promedio global: 22.42 días


In [ ]:
px.pie(df1, names='estado_turno', title='Tasa de pacientes').show()

### Volumen de demanda — tendencia mensual

In [ ]:
# GRÁFICA Volumen de Demanda vs Tendencia Mensual

fig1 = go.Figure()

# Barras de volumen
fig1.add_trace(go.Bar(x=demanda_mensual['anio_mes'], y=demanda_mensual['volumen_demanda'], name='Volumen de demanda', marker_color='#4C78A8', text=demanda_mensual['volumen_demanda'], textposition='outside', hovertemplate='<b>%{x}</b><br>Turnos solicitados: %{y}<extra></extra>'))

# Línea de tendencia
fig1.add_trace(go.Scatter(x=demanda_mensual['anio_mes'], y=demanda_mensual['tendencia'], mode='lines+markers', name=f'Tendencia (pendiente={coef[0]:.2f})', line=dict(color='#E45756', width=3, dash='dash'), marker=dict(size=8), hovertemplate='Tendencia: %{y:.0f}<extra></extra>'))

fig1.update_layout(title=dict(text='<b>Volumen de Demanda vs Tendencia Mensual</b>', font=dict(size=18)), height=500, template='plotly_white', legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1), bargap=0.25, xaxis_title="Mes de solicitud", yaxis_title="N° de turnos solicitados")

fig1.show()

  La demanda mensual es relativamente estable en el rango de 3,764 a 4,313 turnos/mes una vez consolidada la operación (excluyendo jul-25 y jul-26 por ser meses incompletos), con una variación de aproximadamente 14% entre el mes de mayor y menor demanda. Sin embargo, sí existe una tendencia creciente (pendiente = +54.47 turnos/mes), por lo que la planificación de capacidad debería considerar un crecimiento gradual y no una dotación totalmente fija.

### Estado de los turnos y distribución por sede

In [ ]:
px.pie(df1, names='estado_turno', title='Estado de turnos').show()

In [ ]:
px.histogram(df, x = 'sede', text_auto = True).show()

In [ ]:
px.histogram(df, x='sede', text_auto=True, color='estado_turno', barmode='group').show()

In [ ]:
px.histogram(df, x='especialidad', text_auto=True, color='estado_turno', barmode='group').show()

  El ausentismo es el verdadero problema, no la cancelación. La cancelación (6%) puede ser legítima (paciente reprograma). El ausentismo (24%) es la fuga real de capacidad.

  La estabilidad entre sedes sugiere un problema sistémico, no local. Si las 3 sedes tienen el mismo ~30% de no atención, el problema no es de gestión de una sede específica, sino de políticas generales (recordatorios, confirmación de turnos, sobreagendamiento, etc.).

  Oportunidad cuantificable: Si se reduce el ausentismo del 24% al 15% (nivel típico en salud), se recuperarían ~1,500 turnos por sede al año, equivalentes a ~9% más capacidad efectiva sin contratar más personal.

Recomendación adicional: dado que el ausentismo es la palanca principal y que Psiquiatría concentra el mayor problema, las acciones de reducción de ausentismo deberían priorizarse en esa especialidad, mientras que la planificación de capacidad por sede puede mantenerse relativamente uniforme.

#### Carga de trabajo por profesional

In [ ]:
# Calcular días de espera
df['dias_espera'] = (df['fecha_hora_turno'] - df['fecha_solicitud']).dt.total_seconds() / 86400

# Eliminar posibles valores negativos
df = df[df['dias_espera'] >= 0]

print(f"Total de registros válidos: {len(df)}")
print(f"Días de espera promedio global: {df['dias_espera'].mean():.2f} días")
print(f"Mediana: {df['dias_espera'].median():.2f} días")
print(f"Desviación estándar: {df['dias_espera'].std():.2f} días")

Total de registros válidos: 50000
Días de espera promedio global: 22.42 días
Mediana: 22.00 días
Desviación estándar: 12.69 días


In [ ]:
df['semana_solicitud'] = df['fecha_solicitud'].dt.to_period('W').dt.start_time
evol = df.groupby('semana_solicitud')['dias_espera'].mean().reset_index()

fig = px.line(
    evol, x='semana_solicitud', y='dias_espera',
    title='Evolución del Promedio de Días de Espera (por semana de solicitud)',
    labels={'semana_solicitud': 'Semana de solicitud', 'dias_espera': 'Días promedio'}
)
fig.update_traces(line_color='crimson', line_width=2)
fig.update_layout(height=450)
fig.show()

In [ ]:
prof = df.groupby('id_profesional')['dias_espera'].agg(['mean', 'count']).reset_index()
prof = prof[prof['count'] >= 10]  # filtrar muestra significativa
prof.columns = ['Profesional', 'Promedio', 'Cantidad']

top_mayor = prof.nlargest(10, 'Promedio')

fig = px.bar(
    top_mayor, x='Promedio', y='Profesional',
    orientation='h',
    title='Top 10 Profesionales con Mayor Espera Promedio',
    labels={'Promedio': 'Días de espera', 'Profesional': 'Profesional'},
    color='Promedio', color_continuous_scale='Reds',
    text_auto='.1f'
)
fig.update_layout(height=500, showlegend=False)
fig.show()

In [ ]:
resumen = pd.DataFrame({
    'Métrica': ['Promedio global', 'Mediana', 'Mínimo', 'Máximo', 'P25', 'P75', 'P90'],
    'Días de espera': [
        round(df['dias_espera'].mean(), 2),
        round(df['dias_espera'].median(), 2),
        round(df['dias_espera'].min(), 2),
        round(df['dias_espera'].max(), 2),
        round(df['dias_espera'].quantile(0.25), 2),
        round(df['dias_espera'].quantile(0.75), 2),
        round(df['dias_espera'].quantile(0.90), 2),
    ]
})

print("\n=== RESUMEN EJECUTIVO ===")
print(resumen.to_string(index=False))
print(f"\nTotal de turnos analizados: {len(df):,}")


=== RESUMEN EJECUTIVO ===
        Métrica  Días de espera
Promedio global           22.42
        Mediana           22.00
         Mínimo            1.00
         Máximo           44.00
            P25           11.00
            P75           33.00
            P90           40.00

Total de turnos analizados: 50,000


  La espera es homogénea entre especialidades (22.2–22.7 días) y entre los profesionales con mayor espera (22.7–23.1 días), lo que sugiere que el cuello de botella no es una especialidad ni un profesional puntual, sino un problema estructural de capacidad total. El ausentismo, en cambio, sí varía fuertemente por especialidad: Psiquiatría ronda el 35%, más del doble que Oncología (~15%). Por lo tanto, las acciones de reducción de ausentismo deberían priorizarse en Psiquiatría, mientras que la planificación de capacidad debe abordarse a nivel sistémico. (Para confirmar si hay profesionales individuales con ausentismo crítico, se requiere una gráfica adicional de ausentismo por profesional.)

### Insights

  1. Demanda estable, con ligera tendencia creciente
  La demanda mensual es relativamente estable (~4,000 turnos/mes) y sin estacionalidad marcada. Sin embargo, sí existe una tendencia creciente (pendiente ≈ +54 turnos/mes), por lo que la planificación debe considerar crecimiento gradual. La franja de la tarde concentra el 41.9% de la demanda, lo que sí exige refuerzos horarios específicos.

  2. Psiquiatría concentra el riesgo de ausentismo
  34.8% de ausentismo, más del doble que Oncología (14.7%) y muy por encima del promedio general (24.0%). Es la palanca de mayor impacto para recuperar capacidad de agenda.

  3. La espera es estructural, no puntual
  El tiempo de espera (22.2–22.7 días) es casi idéntico entre especialidades, sedes y profesionales. El problema no es un cuello de botella localizado, sino un desajuste sistémico entre demanda y capacidad de agenda. La caída reciente de la espera sugiere que la operación puede responder cuando se ajusta, por lo que parte del problema es de gestión, no solo de capacidad instalada.

### Anomalías y riesgos detectados

  🔴 ALTA — Ausentismo anómalo en Psiquiatría
  34.8% vs 24.0% promedio general — el valor más alto y el único que se aleja claramente del resto de especialidades. Requiere intervención prioritaria.

  🟡 MEDIA — Concentración de ausentismo en pocos profesionales
  Los profesionales MED-12, MED-42 y MED-27 superan el 25.5% de ausentismo, ligeramente por encima del promedio (24.0%). Se recomienda validar con una gráfica de ausentismo por profesional para confirmar si la concentración es significativa o si la diferencia es marginal.

  ⚪ BAJA — Sin desbalance relevante en sede ni cancelaciones
  Volumen por sede (16,550–16,740) y tasa de cancelación (5.7%–6.4%) se mantienen homogéneos. No requieren intervención inmediata. (El análisis por día de la semana requiere gráfica adicional para confirmarse.)

### Siguientes pasos

  1. Reforzar confirmación de turnos en Psiquiatría
  Recordatorios y confirmación previa podrían acercar su ausentismo (34.8%) al promedio (24.0%), liberando cientos de turnos al año. Prioridad alta.

  2. Auditar los profesionales con ausentismo >25%
  MED-12, MED-42 y MED-27 como primeros casos a revisar (agenda, franja horaria, tipo de paciente). Validar primero con una gráfica de ausentismo por profesional para confirmar que la concentración es significativa.

  3. Reforzar dotación en la franja de la tarde
  Concentra 41.9% de la demanda diaria de forma constante — es la franja con mayor riesgo de saturación. Acción operativa inmediata.

  4. Usar 22.4 días de espera como línea base
  Punto de referencia único y comparable para medir el impacto de futuras mejoras de capacidad. No confundir con SLA objetivo: el SLA debería fijarse por debajo del promedio actual (por ejemplo, P50 = 22 días o P25 = 11 días).

  5. Usar estos patrones como insumo del módulo predictivo
  Demanda con tendencia creciente (+54 turnos/mes) + ausentismo concentrado en Psiquiatría + espera homogénea (~22 días) son la base para el modelo de proyección de HealthDemand.
